# Thực nghiệm C: Visual Question Answering (VQA) với Frozen CLIP + Image-Grouped On-the-Fly & RAM LRU Cache

## 1. Tổng quan và Mục tiêu Nghiên cứu

Notebook này triển khai **Thực nghiệm C (Token-Level Cross-Attention)** với kiến trúc tối ưu cho giới hạn ổ đĩa 20 GB của Kaggle (`/kaggle/working/`): **không lưu cache 70–80 GB ra ổ đĩa**, thay vào đó trích xuất trực tiếp (**on-the-fly**) bằng Frozen CLIP kết hợp hai tầng tăng tốc trong bộ nhớ:

1. **Khử trùng lặp ảnh trong từng batch + Gom batch theo ảnh (`ImageGroupedBatchSampler`)**: Trong bộ dữ liệu VQA (~10 câu hỏi/ảnh), các câu hỏi dùng chung ảnh được gom gần nhau theo từng epoch (vẫn xáo trộn ngẫu nhiên thứ tự nhóm ảnh mỗi epoch). Ảnh chỉ bị chạy qua `CLIPVisionModel` **đúng 1 lần cho mỗi ảnh duy nhất trong batch** rồi broadcast sang các câu hỏi liên quan.
2. **Bộ nhớ đệm LRU trên RAM hệ thống (`InMemoryImageFeatureLRU`)**: Lưu các tensor patch `FP16` (`[196, 768]`, ~294 KB/ảnh) ngay trong RAM (tự động giới hạn an toàn theo dung lượng RAM trống, ví dụ tối đa 12 GB trên Kaggle 30 GB RAM $\approx 41,000$ ảnh). Các ảnh đã có trong RAM cache sẽ được tái sử dụng **không cần đọc lại ảnh từ đĩa và không cần chạy lại ViT** ở các epoch tiếp theo.

### Sơ đồ Kiến trúc Tổng thể

```text
                    ON-THE-FLY FROZEN CLIP + RAM LRU (0 GB DISK CACHE)

  Batch câu hỏi [B] (đã gom nhóm theo image_key qua ImageGroupedBatchSampler)
         │
         ├──► Image Keys ──► Kiểm tra InMemoryImageFeatureLRU (RAM, FP16)
         │                        ├── Cache Hit  : Lấy ngay [196, 768] từ RAM (0 I/O, 0 ViT)
         │                        └── Cache Miss : Chỉ nạp & chạy Frozen CLIP Vision
         │                                         cho các ảnh DUY NHẤT chưa có trong RAM
         │                                         ──► Lưu vào RAM LRU ──► Broadcast [B, 196, 768]
         │
         └──► Questions  ──► CLIPTokenizerFast ──► Frozen CLIP Text (no_grad + FP16)
                                                   ──► Token Hidden States [B, L, 512] + Mask [B, L]


                     HUẤN LUYỆN TOKEN-LEVEL CROSS-ATTENTION (TRAINABLE)

     Question Tokens [B, L, 512]                  Image Patches [B, 196, 768]
                 │                                              │
          Text Projection                                Image Projection
         (LayerNorm + Linear)                          (LayerNorm + Linear)
                 │                                              │
                 ▼  Q [B, L, 256]                               ▼  K, V [B, 196, 256]
                 └───────────────► CROSS-ATTENTION ◄────────────┘
                                   (MultiheadAttention, 8 heads)
                                          │
                                  Residual + LayerNorm
                                          │
                                   FFN (256 → 1024 → 256)
                                          │
                                  Residual + LayerNorm
                                  (lặp L_cross = 2 tầng)
                                          │
                              Masked Mean Pooling (bỏ PAD)
                                          │  [B, 256]
                                          ▼
                                   MLP Classifier
                                          │
                                          ▼
                               Answer Logits [B, num_answers]
```

## 2. Thư viện và Phụ thuộc (Dependencies & Imports)

Nạp PyTorch, Hugging Face `transformers` (`CLIPVisionModel`, `CLIPTextModel`, `CLIPImageProcessor`, `CLIPTokenizerFast`), `collections.OrderedDict` cho bộ nhớ đệm RAM LRU, xử lý đa luồng I/O (`ThreadPoolExecutor`) và các thư viện đánh giá/trực quan hóa.

In [ ]:
import os
import sys
import gc
import time
import math
import json
import csv
import ctypes
import zipfile
import random
import collections
from collections import OrderedDict
from concurrent.futures import ThreadPoolExecutor
from typing import Dict, List, Optional, Tuple, Any, Iterator

import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, accuracy_score

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.backends.cudnn as cudnn
from torch.utils.data import Dataset, DataLoader, Sampler
import torchvision.models as models
from torchvision import transforms

try:
    from transformers import (
        CLIPVisionModel,
        CLIPTextModel,
        CLIPImageProcessor,
        CLIPTokenizerFast,
    )
except ImportError as exc:
    raise ImportError(
        "Cần cài đặt thư viện 'transformers' (pip install transformers>=4.38.0) để chạy CLIP."
    ) from exc


def get_available_system_ram_gb() -> float:
    '''Ước lượng RAM hệ thống khả dụng (GB) để giới hạn kích thước LRU cache an toàn.'''
    try:
        import psutil
        return psutil.virtual_memory().available / (1024 ** 3)
    except ImportError:
        pass

    if os.path.exists("/proc/meminfo"):
        try:
            with open("/proc/meminfo", "r", encoding="utf-8") as f:
                for line in f:
                    if line.startswith("MemAvailable:"):
                        kb = int(line.split()[1])
                        return kb / (1024 ** 2)
        except Exception:
            pass

    if os.name == "nt":
        try:
            class MEMORYSTATUSEX(ctypes.Structure):
                _fields_ = [
                    ("dwLength", ctypes.c_ulong),
                    ("dwMemoryLoad", ctypes.c_ulong),
                    ("ullTotalPhys", ctypes.c_ulonglong),
                    ("ullAvailPhys", ctypes.c_ulonglong),
                    ("ullTotalPageFile", ctypes.c_ulonglong),
                    ("ullAvailPageFile", ctypes.c_ulonglong),
                    ("ullTotalVirtual", ctypes.c_ulonglong),
                    ("ullAvailVirtual", ctypes.c_ulonglong),
                    ("sullAvailExtendedVirtual", ctypes.c_ulonglong),
                ]
            stat = MEMORYSTATUSEX()
            stat.dwLength = ctypes.sizeof(MEMORYSTATUSEX)
            ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(stat))
            return stat.ullAvailPhys / (1024 ** 3)
        except Exception:
            pass
    return 8.0


avail_ram_gb = get_available_system_ram_gb()
print(f"Python         : {sys.version.split()[0]}")
print(f"PyTorch        : {torch.__version__}")
print(f"CUDA ready     : {torch.cuda.is_available()}")
print(f"Available RAM  : {avail_ram_gb:.2f} GB")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU device     : {torch.cuda.get_device_name(0)} ({props.total_memory / (1024 ** 3):.2f} GB VRAM)")

## 3. Cấu hình Tái lập & Tham số Trung tâm (Reproducibility & Configuration)

Tập trung toàn bộ cấu hình đường dẫn Kaggle, tham số kiến trúc Cross-Attention, cấu hình bộ nhớ đệm RAM LRU (`RAM_CACHE_MAX_GB`), gom batch theo ảnh (`USE_IMAGE_GROUPED_SAMPLER`) và chế độ smoke test (`DEBUG_MODE`) vào `Path` và `Config`.

> **Khi chuyển từ Smoke Test sang Huấn luyện Đầy đủ**:
> Sửa `Config.DEBUG_MODE = False` ở cell dưới đây và nhấn **Run All**. Notebook sẽ sử dụng **0 GB đĩa** cho cache đặc trưng và chỉ lưu checkpoint mô hình nhẹ (~8 MB) tại `/kaggle/working/best_clip_crossattn.pth`.

In [ ]:
SEED = 42


def set_seed(seed: int = SEED) -> torch.Generator:
    '''Thiết lập seed cố định cho Python, NumPy và PyTorch.'''
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    generator = torch.Generator()
    generator.manual_seed(seed)
    return generator


g = set_seed(SEED)
IS_KAGGLE_ENV = os.path.exists("/kaggle/input") or os.path.exists("/kaggle/working")


class Path:
    # Đường dẫn gốc của dữ liệu trên Kaggle (giữ nguyên cấu trúc gốc)
    BASE_DIR = "/kaggle/input/datasets/lbophc/nckh-vqa"

    # Đường dẫn file JSON câu hỏi
    TRAIN_SAMPLES_JSON = "/kaggle/input/datasets/lbophc/samples/train_samples.json"
    VAL_SAMPLES_JSON = "/kaggle/input/datasets/lbophc/samples/val_samples.json"
    ORIGINAL_JSON = os.path.join(BASE_DIR, "questions/original.json")
    TRAIN_JSON = os.path.join(BASE_DIR, "questions/train.json")
    VAL_JSON = os.path.join(BASE_DIR, "questions/val.json")
    TEST_JSON = os.path.join(BASE_DIR, "questions/test.json")

    # Thư mục chứa hình ảnh
    TRAIN_IMG_DIR = os.path.join(BASE_DIR, "images/train")
    TEST_IMG_DIR = os.path.join(BASE_DIR, "images/test")

    # Đường dẫn vocab gốc (sẽ được kiểm tra tính hợp lệ trước khi tái sử dụng)
    VOCAB_PATH = "/kaggle/input/models/lbophc/vocab/pytorch/default/1/vocab.pth"
    ANS_PATH = "/kaggle/input/models/lbophc/vocab/pytorch/default/1/ans_vocab.pth"

    # Thư mục làm việc ghi được (Kaggle: /kaggle/working)
    WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else os.path.abspath("./kaggle_working")

    # Lưu từ điển câu trả lời được xác thực/tạo từ tập train
    ANS_VOCAB_SAVE_PATH = os.path.join(WORKING_DIR, "ans_vocab_train_only.pth")

    # Checkpoint và artifact cho Thực nghiệm C (không ghi đè best_model.pth của baseline)
    BASELINE_SAVE_PATH = "/kaggle/working/best_model.pth"
    SAVE_PATH = "/kaggle/working/best_clip_crossattn.pth" if os.path.exists("/kaggle/working") else os.path.join(WORKING_DIR, "best_clip_crossattn.pth")
    HISTORY_JSON_PATH = "/kaggle/working/clip_crossattn_history.json" if os.path.exists("/kaggle/working") else os.path.join(WORKING_DIR, "clip_crossattn_history.json")
    HISTORY_PLOT_PATH = "/kaggle/working/clip_crossattn_history.png" if os.path.exists("/kaggle/working") else os.path.join(WORKING_DIR, "clip_crossattn_history.png")
    ATTN_VIZ_DIR = os.path.join(WORKING_DIR, "attention_viz")
    SUBMISSION_PATH = "/kaggle/working/submission.csv" if os.path.exists("/kaggle/working") else os.path.join(WORKING_DIR, "submission.csv")
    SUBMISSION_ZIP_PATH = "/kaggle/working/submission.zip" if os.path.exists("/kaggle/working") else os.path.join(WORKING_DIR, "submission.zip")


class Config:
    # --- Pretrained CLIP Backbone ---
    MODEL_NAME = "openai/clip-vit-base-patch16"
    # Nếu Kaggle tắt Internet, gắn checkpoint cục bộ vào Kaggle Dataset/Model và trỏ đường dẫn này tới thư mục đó
    LOCAL_CLIP_PATH = os.environ.get("CLIP_LOCAL_DIR", "/kaggle/input/clip-vit-base-patch16")

    # --- Token-Level Cross-Attention Architecture ---
    D_MODEL = 256
    NUM_HEADS = 8
    NUM_CROSS_LAYERS = 2
    FFN_DIM = 1024
    DROPOUT = 0.1

    # --- Expected CLIP ViT-B/16 Dimensions ---
    IMAGE_SIZE = 224
    NUM_PATCHES = 196
    VISION_DIM = 768
    TEXT_DIM = 512
    MAX_TEXT_LEN = 77

    # --- Option 2: On-the-Fly + Image-Grouped Deduplication + RAM LRU Cache ---
    RAM_CACHE_ENABLED = True
    # Tự động giới hạn tối đa RAM_CACHE_MAX_GB (hoặc tối đa 55% RAM trống hiện tại)
    RAM_CACHE_MAX_GB = 12.0 if IS_KAGGLE_ENV else 2.0
    USE_IMAGE_GROUPED_SAMPLER = True
    IMAGE_LOAD_THREADS = 4

    # --- Training Hyperparameters ---
    BATCH_SIZE = 64
    EPOCHS = 30
    LEARNING_RATE = 3e-4
    LR = LEARNING_RATE
    WEIGHT_DECAY = 0.01
    GRAD_CLIP_NORM = 1.0
    EARLY_STOPPING_PATIENCE = 6
    WARMUP_RATIO = 0.1
    SEED = SEED
    MAX_LEN = 25  # Chỉ giữ cho tham chiếu lớp Baseline cũ

    # --- DataLoader & Device ---
    # Với Option 2, việc đọc ảnh & cache nằm tập trung trong BatchCLIPFeatureExtractor để khử trùng lặp
    NUM_WORKERS = 0
    PIN_MEMORY = False
    AUTO_SCALE_BATCH_SIZE = True

    # --- Smoke Test / Debug Configuration ---
    DEBUG_MODE = True
    DEBUG_TRAIN_SAMPLES = 64
    DEBUG_VAL_SAMPLES = 32
    DEBUG_TEST_SAMPLES = 32
    ALLOW_LOCAL_SYNTHETIC_FALLBACK = not IS_KAGGLE_ENV

    # --- Optional Modules ---
    RUN_SMOKE_OVERFIT_TEST = True
    RUN_ATTENTION_VIZ = True
    ATTENTION_VIZ_EXAMPLES = 3

    # --- Legacy Vocab References ---
    VOCAB = None
    ANS_VOCAB = None


os.makedirs(Path.WORKING_DIR, exist_ok=True)
print(f"Môi trường Kaggle        : {IS_KAGGLE_ENV}")
print(f"MODEL_NAME               : {Config.MODEL_NAME}")
print(f"DEBUG_MODE               : {Config.DEBUG_MODE}")
print(f"RAM_CACHE_ENABLED        : {Config.RAM_CACHE_ENABLED} (Max {Config.RAM_CACHE_MAX_GB:.1f} GB)")
print(f"USE_IMAGE_GROUPED_SAMPLER: {Config.USE_IMAGE_GROUPED_SAMPLER}")
print(f"SAVE_PATH                : {Path.SAVE_PATH}")

## 4. Kiểm tra Dữ liệu & Xác thực Đường dẫn (Dataset Inspection & Path Validation)

Kiểm tra tệp JSON (`train.json`, `val.json`, `test.json`), xác minh ảnh validation nằm trong `TRAIN_IMG_DIR`, xây dựng/xác thực `ans_vocab` chỉ từ tập train (tránh rò rỉ nhãn từ val/test) và áp dụng chế độ `DEBUG_MODE` nếu đang bật.

In [ ]:
def load_torch_file(file_path: str, map_location: Any = "cpu") -> Any:
    '''Nạp file .pth an toàn tương thích các phiên bản PyTorch.'''
    try:
        return torch.load(file_path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(file_path, map_location=map_location)


def load_vqa_json(json_path: str, is_test: bool = False) -> List[Dict[str, Any]]:
    '''Đọc file JSON câu hỏi theo định dạng thực tế của dataset.'''
    with open(json_path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    if is_test:
        if "questions" in raw and isinstance(raw["questions"], list):
            return raw["questions"]
        if "root" in raw and "questions" in raw["root"]:
            return raw["root"]["questions"]
    else:
        if "root" in raw and "questions" in raw["root"]:
            return raw["root"]["questions"]
        if "questions" in raw and isinstance(raw["questions"], list):
            return raw["questions"]

    raise KeyError(f"Cấu trúc JSON không khớp định dạng kỳ vọng tại {json_path}. Keys: {list(raw.keys())}")


def create_local_synthetic_dataset(root_dir: str) -> Dict[str, str]:
    '''Tạo bộ dữ liệu vi mô giả lập chỉ phục vụ kiểm thử khi chạy cục bộ ngoài Kaggle.'''
    syn_dir = os.path.join(root_dir, "synthetic_vqa_debug")
    q_dir = os.path.join(syn_dir, "questions")
    train_img_dir = os.path.join(syn_dir, "images", "train")
    test_img_dir = os.path.join(syn_dir, "images", "test")
    os.makedirs(q_dir, exist_ok=True)
    os.makedirs(train_img_dir, exist_ok=True)
    os.makedirs(test_img_dir, exist_ok=True)

    colors = [("red", (220, 40, 40)), ("blue", (40, 80, 220)), ("green", (40, 180, 60)), ("yellow", (230, 200, 40))]
    shapes = ["cube", "sphere", "cylinder"]

    for idx in range(12):
        fpath = os.path.join(train_img_dir, f"syn_train_{idx:04d}.png")
        if not os.path.exists(fpath):
            _, rgb = colors[idx % len(colors)]
            Image.new("RGB", (256, 256), rgb).save(fpath)

    for idx in range(4):
        fpath = os.path.join(test_img_dir, f"syn_test_{idx:04d}.png")
        if not os.path.exists(fpath):
            _, rgb = colors[idx % len(colors)]
            Image.new("RGB", (256, 256), rgb).save(fpath)

    train_questions = []
    for i in range(64):
        c_name, _ = colors[i % len(colors)]
        s_name = shapes[i % len(shapes)]
        if i % 2 == 0:
            q_text = f"What color is the {s_name} in the scene?"
            ans = c_name
        else:
            q_text = f"How many {c_name} objects are visible?"
            ans = str((i % 3) + 1)
        train_questions.append({
            "qid": 10000 + i,
            "image_filename": f"syn_train_{(i % 12):04d}.png",
            "question": q_text,
            "answer": ans,
        })

    val_questions = []
    for i in range(32):
        c_name, _ = colors[i % len(colors)]
        s_name = shapes[i % len(shapes)]
        val_questions.append({
            "qid": 20000 + i,
            "image_filename": f"syn_train_{(i % 12):04d}.png",
            "question": f"What color is the {s_name} in the scene?" if i % 2 == 0 else f"How many {c_name} objects are visible?",
            "answer": c_name if i % 2 == 0 else str((i % 3) + 1),
        })

    test_questions = []
    for i in range(16):
        test_questions.append({
            "qid": 30000 + i,
            "image_filename": f"syn_test_{(i % 4):04d}.png",
            "question": f"What color is the {shapes[i % len(shapes)]} in the scene?",
        })

    train_json = os.path.join(q_dir, "train.json")
    val_json = os.path.join(q_dir, "val.json")
    test_json = os.path.join(q_dir, "test.json")
    with open(train_json, "w", encoding="utf-8") as f:
        json.dump({"root": {"questions": train_questions}}, f, indent=2)
    with open(val_json, "w", encoding="utf-8") as f:
        json.dump({"root": {"questions": val_questions}}, f, indent=2)
    with open(test_json, "w", encoding="utf-8") as f:
        json.dump({"questions": test_questions}, f, indent=2)

    return {
        "BASE_DIR": syn_dir,
        "TRAIN_JSON": train_json,
        "VAL_JSON": val_json,
        "TEST_JSON": test_json,
        "TRAIN_IMG_DIR": train_img_dir,
        "TEST_IMG_DIR": test_img_dir,
    }


def validate_or_build_ans_vocab(
    full_train_records: List[Dict[str, Any]],
    existing_ans_path: str,
    save_path: str,
) -> Dict[str, int]:
    '''Xác thực ans_vocab chỉ được dựng từ tập train; nếu sai lệch thì dựng lại từ tập train.'''
    train_answers = sorted({str(rec["answer"]) for rec in full_train_records if "answer" in rec})
    expected_vocab = {ans: idx for idx, ans in enumerate(train_answers)}

    if existing_ans_path and os.path.exists(existing_ans_path):
        loaded = load_torch_file(existing_ans_path, map_location="cpu")
        if isinstance(loaded, dict):
            values_sorted = sorted(loaded.values()) if loaded else []
            if set(loaded.keys()) == set(train_answers) and values_sorted == list(range(len(loaded))):
                print(f"[ANS_VOCAB] Hợp lệ từ {existing_ans_path} ({len(loaded)} lớp, chỉ từ tập train).")
                torch.save(loaded, save_path)
                return loaded
            print(
                f"[ANS_VOCAB] CẢNH BÁO: {existing_ans_path} không khớp chặt chẽ tập train. "
                "Đang dựng lại hoàn toàn từ train_records để tránh rò rỉ nhãn..."
            )

    torch.save(expected_vocab, save_path)
    print(f"[ANS_VOCAB] Đã xây dựng và lưu mới từ tập train ({len(expected_vocab)} lớp) tại: {save_path}")
    return expected_vocab


def verify_split_images_exist(
    records: List[Dict[str, Any]],
    img_dir: str,
    split_name: str,
    sample_check_limit: int = 300,
) -> int:
    unique_names = list(dict.fromkeys(rec["image_filename"] for rec in records))
    to_check = unique_names[:sample_check_limit]
    missing = [name for name in to_check if not os.path.exists(os.path.join(img_dir, name))]
    print(
        f"[{split_name.upper()}] Ảnh duy nhất: {len(unique_names):,} | "
        f"Kiểm tra mẫu {len(to_check) - len(missing)}/{len(to_check)} tồn tại tại {img_dir}"
    )
    if missing:
        raise FileNotFoundError(f"Thiếu ảnh cho split '{split_name}' trong {img_dir}: {missing[:5]}")
    return len(unique_names)


def inspect_and_validate_dataset() -> Dict[str, Any]:
    required_paths = [
        Path.BASE_DIR,
        Path.TRAIN_JSON,
        Path.VAL_JSON,
        Path.TEST_JSON,
        Path.TRAIN_IMG_DIR,
        Path.TEST_IMG_DIR,
    ]
    missing_paths = [p for p in required_paths if not os.path.exists(p)]

    if missing_paths:
        if Config.ALLOW_LOCAL_SYNTHETIC_FALLBACK and not IS_KAGGLE_ENV:
            print(
                "[LOCAL SMOKE MODE] Không tìm thấy dữ liệu Kaggle tại "
                f"{Path.BASE_DIR}. Đang tạo dữ liệu giả lập cục bộ để kiểm thử..."
            )
            syn = create_local_synthetic_dataset(Path.WORKING_DIR)
            Path.BASE_DIR = syn["BASE_DIR"]
            Path.TRAIN_JSON = syn["TRAIN_JSON"]
            Path.VAL_JSON = syn["VAL_JSON"]
            Path.TEST_JSON = syn["TEST_JSON"]
            Path.TRAIN_IMG_DIR = syn["TRAIN_IMG_DIR"]
            Path.TEST_IMG_DIR = syn["TEST_IMG_DIR"]
            using_synthetic = True
        else:
            raise FileNotFoundError("Thiếu đường dẫn dataset trên Kaggle:\n" + "\n".join(missing_paths))
    else:
        using_synthetic = False

    full_train = load_vqa_json(Path.TRAIN_JSON, is_test=False)
    full_val = load_vqa_json(Path.VAL_JSON, is_test=False)
    full_test = load_vqa_json(Path.TEST_JSON, is_test=True)

    print("\n--- Thống kê Bản ghi Gốc ---")
    print(f"  Train records : {len(full_train):,}")
    print(f"  Val records   : {len(full_val):,}")
    print(f"  Test records  : {len(full_test):,}")

    n_train_imgs = verify_split_images_exist(full_train, Path.TRAIN_IMG_DIR, "train")
    n_val_imgs = verify_split_images_exist(full_val, Path.TRAIN_IMG_DIR, "val (TRAIN_IMG_DIR)")
    n_test_imgs = verify_split_images_exist(full_test, Path.TEST_IMG_DIR, "test")

    ans_vocab = validate_or_build_ans_vocab(full_train, Path.ANS_PATH, Path.ANS_VOCAB_SAVE_PATH)
    Config.ANS_VOCAB = ans_vocab
    if os.path.exists(Path.VOCAB_PATH):
        Config.VOCAB = load_torch_file(Path.VOCAB_PATH, map_location="cpu")

    val_valid = [r for r in full_val if str(r.get("answer")) in ans_vocab]
    val_oov_count = len(full_val) - len(val_valid)
    if val_oov_count > 0:
        print(f"[CẢNH BÁO] Loại {val_oov_count} mẫu val ngoài train ans_vocab.")

    if Config.DEBUG_MODE:
        train_records = full_train[:Config.DEBUG_TRAIN_SAMPLES]
        val_records = val_valid[:Config.DEBUG_VAL_SAMPLES]
        test_records = full_test[:Config.DEBUG_TEST_SAMPLES]
        print(f"\n[DEBUG_MODE=True] Cắt mẫu: train={len(train_records)}, val={len(val_records)}, test={len(test_records)}")
    else:
        train_records = full_train
        val_records = val_valid
        test_records = full_test
        print(f"\n[FULL DATASET MODE] Toàn bộ: train={len(train_records):,}, val={len(val_records):,}, test={len(test_records):,}")

    print("  Train sample:", train_records[0])
    print("  Val sample  :", val_records[0])
    print("  Test sample :", test_records[0])

    return {
        "train_records": train_records,
        "val_records": val_records,
        "test_records": test_records,
        "ans_vocab": ans_vocab,
        "unique_img_counts": {"train": n_train_imgs, "val": n_val_imgs, "test": n_test_imgs},
        "using_synthetic": using_synthetic,
    }


dataset_info = inspect_and_validate_dataset()

## 5. Nạp Frozen CLIP Encoders & Kiểm tra Kích thước Tensor (Feature-Shape Tests)

Tải checkpoint `openai/clip-vit-base-patch16`:
- Đóng băng 100% tham số của `CLIPVisionModel` và `CLIPTextModel` (`requires_grad=False`) và đặt `.eval()`.
- Xác minh đầu ra Vision (`last_hidden_state[:, 1:, :]`) có kích thước `[B, 196, 768]` (bỏ `CLS` token) và Text (`last_hidden_state`) có kích thước `[B, L, 512]`.

> **Nếu Kaggle tắt Internet**: Upload thư mục checkpoint `openai/clip-vit-base-patch16` lên Kaggle Dataset/Model và gán đường dẫn vào `Config.LOCAL_CLIP_PATH` (không tự ý đổi sang mô hình khác).

In [ ]:
def load_frozen_clip_encoders(
    model_name: str = Config.MODEL_NAME,
    local_path: Optional[str] = Config.LOCAL_CLIP_PATH,
    device: Optional[torch.device] = None,
) -> Tuple[CLIPVisionModel, CLIPTextModel, CLIPImageProcessor, CLIPTokenizerFast]:
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if local_path and os.path.isdir(local_path):
        source = local_path
        load_kwargs = {"local_files_only": True}
        print(f"[CLIP] Nạp từ checkpoint cục bộ: {source}")
    else:
        source = model_name
        load_kwargs = {}

    try:
        image_processor = CLIPImageProcessor.from_pretrained(source, **load_kwargs)
        tokenizer = CLIPTokenizerFast.from_pretrained(source, **load_kwargs)
        vision_model = CLIPVisionModel.from_pretrained(source, **load_kwargs)
        text_model = CLIPTextModel.from_pretrained(source, **load_kwargs)
    except Exception as exc:
        raise RuntimeError(
            f"Không thể nạp CLIP checkpoint '{source}'.\n"
            "Nếu Kaggle đang TẮT Internet: hãy tải trước 'openai/clip-vit-base-patch16', "
            f"gắn dưới dạng Kaggle Dataset/Model và đặt Config.LOCAL_CLIP_PATH (hiện tại: '{local_path}')."
        ) from exc

    for m in (vision_model, text_model):
        m.eval()
        for param in m.parameters():
            param.requires_grad = False
        m.to(device)

    return vision_model, text_model, image_processor, tokenizer


def verify_clip_encoder_shapes(
    vision_model: CLIPVisionModel,
    text_model: CLIPTextModel,
    image_processor: CLIPImageProcessor,
    tokenizer: CLIPTokenizerFast,
    sample_record: Dict[str, Any],
    sample_img_dir: str,
    device: torch.device,
) -> None:
    assert all(not p.requires_grad for p in vision_model.parameters()), "Vision encoder chưa được đóng băng!"
    assert all(not p.requires_grad for p in text_model.parameters()), "Text encoder chưa được đóng băng!"
    assert not vision_model.training and not text_model.training

    img_path = os.path.join(sample_img_dir, sample_record["image_filename"])
    with Image.open(img_path) as im:
        pix = image_processor(images=[im.convert("RGB")], return_tensors="pt")["pixel_values"].to(device)

    enc_txt = tokenizer(
        [sample_record["question"]],
        padding=True,
        truncation=True,
        max_length=min(Config.MAX_TEXT_LEN, tokenizer.model_max_length),
        return_tensors="pt",
    )
    input_ids = enc_txt["input_ids"].to(device)
    attention_mask = enc_txt["attention_mask"].to(device)

    with torch.no_grad():
        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            v_out = vision_model(pixel_values=pix, return_dict=True)
            patch_tokens = v_out.last_hidden_state[:, 1:, :]
            t_out = text_model(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
            text_tokens = t_out.last_hidden_state

    assert patch_tokens.shape == (1, Config.NUM_PATCHES, Config.VISION_DIM), f"Sai shape ảnh: {patch_tokens.shape}"
    assert text_tokens.ndim == 3 and text_tokens.shape[:2] == input_ids.shape
    assert text_tokens.shape[2] == Config.TEXT_DIM, f"Sai shape text: {text_tokens.shape}"

    print("[CHECK PASS] Frozen CLIP Encoders hợp lệ:")
    print(f"  - Vision patch tokens : {tuple(patch_tokens.shape)} (bỏ CLS)")
    print(f"  - Text token features : {tuple(text_tokens.shape)}")
    print(f"  - Attention mask      : {tuple(attention_mask.shape)} (valid={int(attention_mask.sum().item())})")
    print("  - Trainable CLIP params: 0")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vision_model, text_model, image_processor, tokenizer = load_frozen_clip_encoders(
    model_name=Config.MODEL_NAME,
    local_path=Config.LOCAL_CLIP_PATH,
    device=device,
)
verify_clip_encoder_shapes(
    vision_model=vision_model,
    text_model=text_model,
    image_processor=image_processor,
    tokenizer=tokenizer,
    sample_record=dataset_info["train_records"][0],
    sample_img_dir=Path.TRAIN_IMG_DIR,
    device=device,
)

## 6. Bộ nhớ đệm RAM LRU & Trích xuất CLIP Khử trùng lặp Theo Batch (In-Memory Deduplicated Extractor)

Thay vì ghi 70–80 GB ra đĩa, `InMemoryImageFeatureLRU` và `BatchCLIPFeatureExtractor` thực hiện:
- **Trong mỗi batch**: Tìm các `image_key` chưa có trong RAM cache, khử trùng lặp thành danh sách ảnh duy nhất (`unique_missing_keys`), đọc song song và chạy `CLIPVisionModel` **chỉ 1 lần cho mỗi ảnh duy nhất**, lưu tensor `FP16` `[196, 768]` vào RAM LRU rồi broadcast cho toàn bộ các câu hỏi dùng ảnh đó trong batch.
- **Giữa các epoch**: Các ảnh còn nằm trong giới hạn `RAM_CACHE_MAX_GB` sẽ **Cache Hit 100%** — bỏ qua cả thao tác đọc ảnh trên đĩa lẫn forward qua ViT.
- Toàn bộ CLIP chạy trong `with torch.no_grad():` + `AMP FP16` nên không lưu computation graph trên VRAM.

In [ ]:
def make_image_key(img_dir_tag: str, image_filename: str) -> str:
    clean_name = str(image_filename).replace("\\", "/")
    return f"{img_dir_tag}::{clean_name}"


class InMemoryImageFeatureLRU:
    '''Bộ nhớ đệm LRU trong RAM lưu đặc trưng patch FP16 [196, 768] theo image_key.'''

    def __init__(self, max_gb: float = Config.RAM_CACHE_MAX_GB, enabled: bool = Config.RAM_CACHE_ENABLED):
        self.enabled = enabled
        self.bytes_per_image = Config.NUM_PATCHES * Config.VISION_DIM * 2  # FP16 = 2 bytes

        avail_gb = get_available_system_ram_gb()
        safe_cap_gb = min(max_gb, max(0.25, avail_gb * 0.55))
        self.max_bytes = int(safe_cap_gb * (1024 ** 3)) if enabled else 0
        self.max_items = self.max_bytes // self.bytes_per_image if self.bytes_per_image > 0 else 0

        self._cache: OrderedDict[str, torch.Tensor] = OrderedDict()
        self.hits = 0
        self.misses = 0
        self.evictions = 0

    def get(self, key: str) -> Optional[torch.Tensor]:
        if not self.enabled or self.max_items <= 0:
            self.misses += 1
            return None
        val = self._cache.get(key)
        if val is not None:
            self._cache.move_to_end(key)
            self.hits += 1
            return val
        self.misses += 1
        return None

    def put(self, key: str, tensor_fp16_cpu: torch.Tensor) -> None:
        if not self.enabled or self.max_items <= 0:
            return
        if key in self._cache:
            self._cache.move_to_end(key)
            return
        while len(self._cache) >= self.max_items:
            self._cache.popitem(last=False)
            self.evictions += 1
        self._cache[key] = tensor_fp16_cpu

    def reset_stats(self) -> None:
        self.hits = 0
        self.misses = 0

    @property
    def hit_rate(self) -> float:
        total = self.hits + self.misses
        return (self.hits / total) if total > 0 else 0.0

    @property
    def used_mb(self) -> float:
        return (len(self._cache) * self.bytes_per_image) / (1024 ** 2)

    def __len__(self) -> int:
        return len(self._cache)


class BatchCLIPFeatureExtractor:
    '''Trích xuất đặc trưng CLIP cho từng batch với khử trùng lặp ảnh + RAM LRU + Text on-the-fly.'''

    def __init__(
        self,
        vision_model: CLIPVisionModel,
        text_model: CLIPTextModel,
        image_processor: CLIPImageProcessor,
        tokenizer: CLIPTokenizerFast,
        ram_lru: InMemoryImageFeatureLRU,
        device: torch.device,
    ):
        self.vision_model = vision_model
        self.text_model = text_model
        self.image_processor = image_processor
        self.tokenizer = tokenizer
        self.ram_lru = ram_lru
        self.device = device
        self.max_text_len = min(Config.MAX_TEXT_LEN, int(getattr(tokenizer, "model_max_length", Config.MAX_TEXT_LEN)))

        # Bộ đếm thống kê để theo dõi hiệu quả tiết kiệm tính toán
        self.total_image_requests = 0
        self.total_vit_forwards = 0
        self.total_text_forwards = 0

    def reset_counters(self) -> None:
        self.total_image_requests = 0
        self.total_vit_forwards = 0
        self.total_text_forwards = 0
        self.ram_lru.reset_stats()

    @staticmethod
    def _load_rgb_image(path: str) -> Image.Image:
        with Image.open(path) as im:
            return im.convert("RGB")

    def _load_images_parallel(self, paths: List[str]) -> List[Image.Image]:
        if len(paths) <= 4 or Config.IMAGE_LOAD_THREADS <= 1:
            return [self._load_rgb_image(p) for p in paths]
        with ThreadPoolExecutor(max_workers=min(Config.IMAGE_LOAD_THREADS, len(paths))) as ex:
            return list(ex.map(self._load_rgb_image, paths))

    def get_batch_image_features(self, image_keys: List[str], image_paths: List[str]) -> torch.Tensor:
        '''Lấy tensor [B, 196, 768] trên device, chỉ chạy CLIP Vision cho các ảnh duy nhất chưa cache.'''
        self.total_image_requests += len(image_keys)
        resolved: Dict[str, torch.Tensor] = {}
        missing_keys: List[str] = []
        missing_paths: List[str] = []
        seen_missing = set()

        for key, path in zip(image_keys, image_paths):
            if key in resolved:
                continue
            cached = self.ram_lru.get(key)
            if cached is not None:
                resolved[key] = cached
            elif key not in seen_missing:
                seen_missing.add(key)
                missing_keys.append(key)
                missing_paths.append(path)

        if missing_keys:
            pil_images = self._load_images_parallel(missing_paths)
            pix = self.image_processor(images=pil_images, return_tensors="pt")["pixel_values"].to(
                self.device, non_blocking=True
            )
            with torch.no_grad():
                with torch.amp.autocast("cuda", enabled=(self.device.type == "cuda")):
                    v_out = self.vision_model(pixel_values=pix, return_dict=True)
                    patches = v_out.last_hidden_state[:, 1:, :]  # [U, 196, 768]

            patches_cpu_fp16 = patches.detach().to(dtype=torch.float16).cpu()
            self.total_vit_forwards += len(missing_keys)
            for idx, m_key in enumerate(missing_keys):
                feat = patches_cpu_fp16[idx]
                self.ram_lru.put(m_key, feat)
                resolved[m_key] = feat

        batch_cpu = torch.stack([resolved[k] for k in image_keys], dim=0)
        return batch_cpu.to(self.device, non_blocking=True)

    def get_batch_text_features(self, questions: List[str]) -> Tuple[torch.Tensor, torch.Tensor]:
        '''Tokenize và chạy Frozen CLIP Text cho batch câu hỏi -> ([B, L, 512], [B, L] bool).'''
        enc = self.tokenizer(
            questions,
            padding=True,
            truncation=True,
            max_length=self.max_text_len,
            return_tensors="pt",
        )
        input_ids = enc["input_ids"].to(self.device, non_blocking=True)
        attention_mask = enc["attention_mask"].to(self.device, non_blocking=True)

        with torch.no_grad():
            with torch.amp.autocast("cuda", enabled=(self.device.type == "cuda")):
                t_out = self.text_model(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
                q_feats = t_out.last_hidden_state.detach()

        self.total_text_forwards += len(questions)
        return q_feats, attention_mask.bool()

    def extract_batch(
        self,
        image_keys: List[str],
        image_paths: List[str],
        questions: List[str],
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        img_feats = self.get_batch_image_features(image_keys, image_paths)
        q_feats, q_mask = self.get_batch_text_features(questions)
        return img_feats, q_feats, q_mask


ram_lru_cache = InMemoryImageFeatureLRU(
    max_gb=Config.RAM_CACHE_MAX_GB,
    enabled=Config.RAM_CACHE_ENABLED,
)
clip_extractor = BatchCLIPFeatureExtractor(
    vision_model=vision_model,
    text_model=text_model,
    image_processor=image_processor,
    tokenizer=tokenizer,
    ram_lru=ram_lru_cache,
    device=device,
)
print(
    f"[RAM LRU READY] Dung lượng tối đa: {ram_lru_cache.max_bytes / (1024**3):.2f} GB "
    f"(~{ram_lru_cache.max_items:,} ảnh duy nhất FP16 | 0 MB đĩa)"
)

## 7. Kiểm chứng Khử trùng lặp Ảnh & Tốc độ Trích xuất On-the-Fly

Kiểm tra trực tiếp cơ chế khử trùng lặp và RAM LRU trên một nhóm mẫu thực tế:
- Lần 1 (cold cache): chỉ chạy `CLIPVisionModel` đúng số lượng ảnh duy nhất thay vì số câu hỏi.
- Lần 2 (warm cache): số lần gọi `CLIPVisionModel` mới bằng `0` (100% lấy từ RAM LRU).

In [ ]:
def verify_dedup_and_ram_lru(extractor: BatchCLIPFeatureExtractor, sample_records: List[Dict[str, Any]], img_dir: str) -> None:
    extractor.reset_counters()
    probe = sample_records[: min(32, len(sample_records))]
    keys = [make_image_key("train_img", r["image_filename"]) for r in probe]
    paths = [os.path.join(img_dir, r["image_filename"]) for r in probe]
    questions = [r["question"] for r in probe]
    unique_count = len(set(keys))

    # Lần 1: Cold RAM cache
    t0 = time.perf_counter()
    img_1, q_1, mask_1 = extractor.extract_batch(keys, paths, questions)
    t_cold = time.perf_counter() - t0
    vit_calls_cold = extractor.total_vit_forwards

    assert vit_calls_cold == unique_count, f"Kỳ vọng {unique_count} lần chạy ViT, thực tế {vit_calls_cold}"
    assert img_1.shape == (len(probe), Config.NUM_PATCHES, Config.VISION_DIM)
    assert q_1.ndim == 3 and q_1.shape[0] == len(probe) and q_1.shape[2] == Config.TEXT_DIM
    assert mask_1.shape == q_1.shape[:2]

    # Lần 2: Warm RAM cache -> 0 lần chạy ViT
    t1 = time.perf_counter()
    img_2, q_2, mask_2 = extractor.extract_batch(keys, paths, questions)
    t_warm = time.perf_counter() - t1
    vit_calls_warm = extractor.total_vit_forwards - vit_calls_cold

    assert vit_calls_warm == 0, f"Kỳ vọng 0 lần chạy ViT ở lần 2, thực tế {vit_calls_warm}"
    assert torch.equal(img_1, img_2), "Đặc trưng ảnh từ RAM LRU không khớp lần trích xuất đầu!"

    print("[DEDUP & LRU PASS]")
    print(f"  - Câu hỏi trong batch    : {len(probe)} | Ảnh duy nhất: {unique_count}")
    print(f"  - Lần 1 (cold) ViT calls : {vit_calls_cold} ({t_cold*1000:.1f} ms)")
    print(f"  - Lần 2 (warm) ViT calls : {vit_calls_warm} ({t_warm*1000:.1f} ms, RAM hit rate={extractor.ram_lru.hit_rate*100:.1f}%)")
    extractor.reset_counters()


verify_dedup_and_ram_lru(clip_extractor, dataset_info["train_records"], Path.TRAIN_IMG_DIR)

## 8. Thống kê Dự toán Bộ nhớ Đệm RAM & Tỷ lệ Tái sử dụng Ảnh

Tính số ảnh duy nhất trên toàn bộ split hiện tại (`train`, `val`, `test`) và tỷ lệ bao phủ mà `InMemoryImageFeatureLRU` có thể giữ trong RAM (so sánh với **0 MB** trên ổ đĩa).

In [ ]:
def print_memory_budget_report(dataset_info: Dict[str, Any], ram_lru: InMemoryImageFeatureLRU) -> None:
    tr = dataset_info["train_records"]
    va = dataset_info["val_records"]
    te = dataset_info["test_records"]

    tr_uniq = len({r["image_filename"] for r in tr})
    va_uniq = len({r["image_filename"] for r in va})
    te_uniq = len({r["image_filename"] for r in te})
    total_uniq = len(
        {("train_img", r["image_filename"]) for r in tr + va}
        | {("test_img", r["image_filename"]) for r in te}
    )
    total_q = len(tr) + len(va) + len(te)

    ram_needed_all_gb = (total_uniq * ram_lru.bytes_per_image) / (1024 ** 3)
    coverage = min(100.0, (ram_lru.max_items / max(total_uniq, 1)) * 100.0)

    print("=== BÁO CÁO BỘ NHỚ & TÁI SỬ DỤNG ẢNH (OPTION 2) ===")
    print(f"  Train : {len(tr):,} câu hỏi / {tr_uniq:,} ảnh duy nhất (~{len(tr)/max(tr_uniq,1):.1f} q/ảnh)")
    print(f"  Val   : {len(va):,} câu hỏi / {va_uniq:,} ảnh duy nhất (~{len(va)/max(va_uniq,1):.1f} q/ảnh)")
    print(f"  Test  : {len(te):,} câu hỏi / {te_uniq:,} ảnh duy nhất (~{len(te)/max(te_uniq,1):.1f} q/ảnh)")
    print(f"  Dung lượng đĩa dùng để cache : 0.00 MB (On-the-fly)")
    print(f"  RAM cần để cache 100% ảnh    : {ram_needed_all_gb:.2f} GB ({total_uniq:,} ảnh FP16)")
    print(f"  Giới hạn RAM LRU hiện tại    : {ram_lru.max_bytes / (1024**3):.2f} GB (~{ram_lru.max_items:,} ảnh, phủ {coverage:.1f}% tổng ảnh)")


print_memory_budget_report(dataset_info, ram_lru_cache)

## 9. On-the-Fly Dataset, Image-Grouped Sampler & Bảo tồn Baseline Cũ

- `OnTheFlyVQADataset`: Chỉ lưu metadata nhẹ (`image_key`, `image_path`, `question`, `label`/`qid`), không đọc PNG dư thừa nếu ảnh đã nằm trong RAM LRU.
- `ImageGroupedBatchSampler`: Xáo trộn thứ tự các ảnh duy nhất ở mỗi epoch và xếp các câu hỏi của cùng 1 ảnh vào sát nhau để mỗi batch `64` câu hỏi chỉ chứa ~`6–8` ảnh duy nhất thay vì `64` ảnh.
- Cell tham chiếu ngay bên dưới bảo tồn nguyên vẹn `VQADataset`, `Data_Loader`, `Model_ResNet_LSTM`, `Model_ViT_Cross_attn` để đối chiếu về sau (không tự động thực thi).

In [ ]:
class OnTheFlyVQADataset(Dataset):
    '''Dataset VQA nhẹ trả về khóa ảnh, đường dẫn, câu hỏi và nhãn/qid.'''

    def __init__(
        self,
        records: List[Dict[str, Any]],
        img_dir: str,
        img_dir_tag: str,
        ans_vocab: Optional[Dict[str, int]] = None,
        is_test: bool = False,
    ):
        self.records = records
        self.img_dir = img_dir
        self.img_dir_tag = img_dir_tag
        self.ans_vocab = ans_vocab
        self.is_test = is_test

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        rec = self.records[idx]
        fname = rec["image_filename"]
        item: Dict[str, Any] = {
            "image_key": make_image_key(self.img_dir_tag, fname),
            "image_path": os.path.join(self.img_dir, fname),
            "question": rec["question"],
        }
        if self.is_test:
            item["qid"] = rec["qid"]
        else:
            item["label"] = self.ans_vocab[str(rec["answer"])]
        return item


class ImageGroupedBatchSampler(Sampler[List[int]]):
    '''
    Gom các câu hỏi có chung image_key vào cùng batch để tối đa hóa khử trùng lặp ViT,
    đồng thời xáo trộn ngẫu nhiên thứ tự ảnh và câu hỏi bên trong mỗi epoch.
    '''

    def __init__(self, records: List[Dict[str, Any]], batch_size: int, seed: int = SEED, shuffle: bool = True):
        self.batch_size = batch_size
        self.seed = seed
        self.shuffle = shuffle
        self.epoch = 0

        groups_dict: Dict[str, List[int]] = collections.defaultdict(list)
        for idx, rec in enumerate(records):
            groups_dict[rec["image_filename"]].append(idx)
        self.groups: List[List[int]] = list(groups_dict.values())
        self.total_samples = len(records)

    def set_epoch(self, epoch: int) -> None:
        self.epoch = epoch

    def __len__(self) -> int:
        return (self.total_samples + self.batch_size - 1) // self.batch_size

    def __iter__(self) -> Iterator[List[int]]:
        if self.shuffle:
            rng = random.Random(self.seed + self.epoch)
            group_order = list(range(len(self.groups)))
            rng.shuffle(group_order)
            ordered_indices: List[int] = []
            for g_idx in group_order:
                idxs = list(self.groups[g_idx])
                rng.shuffle(idxs)
                ordered_indices.extend(idxs)
        else:
            ordered_indices = [idx for grp in self.groups for idx in grp]

        for start in range(0, len(ordered_indices), self.batch_size):
            yield ordered_indices[start : start + self.batch_size]


def on_the_fly_collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    collated: Dict[str, Any] = {
        "image_keys": [b["image_key"] for b in batch],
        "image_paths": [b["image_path"] for b in batch],
        "questions": [b["question"] for b in batch],
    }
    if "label" in batch[0]:
        collated["label"] = torch.tensor([b["label"] for b in batch], dtype=torch.long)
    if "qid" in batch[0]:
        collated["qid"] = [b["qid"] for b in batch]
    return collated


def build_on_the_fly_dataloaders(
    dataset_info: Dict[str, Any],
    batch_size: int,
) -> Tuple[OnTheFlyVQADataset, OnTheFlyVQADataset, OnTheFlyVQADataset, DataLoader, DataLoader, DataLoader]:
    ans_vocab = dataset_info["ans_vocab"]
    train_ds = OnTheFlyVQADataset(dataset_info["train_records"], Path.TRAIN_IMG_DIR, "train_img", ans_vocab=ans_vocab, is_test=False)
    val_ds = OnTheFlyVQADataset(dataset_info["val_records"], Path.TRAIN_IMG_DIR, "train_img", ans_vocab=ans_vocab, is_test=False)
    test_ds = OnTheFlyVQADataset(dataset_info["test_records"], Path.TEST_IMG_DIR, "test_img", ans_vocab=None, is_test=True)

    if Config.USE_IMAGE_GROUPED_SAMPLER:
        train_sampler = ImageGroupedBatchSampler(dataset_info["train_records"], batch_size=batch_size, seed=Config.SEED, shuffle=True)
        train_loader = DataLoader(train_ds, batch_sampler=train_sampler, num_workers=0, collate_fn=on_the_fly_collate_fn)
    else:
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, generator=g, num_workers=0, collate_fn=on_the_fly_collate_fn)

    # Val dùng grouped không shuffle để tối ưu cache; Test giữ nguyên thứ tự gốc 100% để khớp qid
    val_sampler = ImageGroupedBatchSampler(dataset_info["val_records"], batch_size=batch_size, seed=Config.SEED, shuffle=False)
    val_loader = DataLoader(val_ds, batch_sampler=val_sampler, num_workers=0, collate_fn=on_the_fly_collate_fn)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0, collate_fn=on_the_fly_collate_fn)

    return train_ds, val_ds, test_ds, train_loader, val_loader, test_loader


print("Đã định nghĩa OnTheFlyVQADataset & ImageGroupedBatchSampler thành công.")

In [ ]:
# ==============================================================================
# [THAM CHIẾU BASELINE CŨ - KHÔNG TỰ ĐỘNG THỰC THI]
# Bảo tồn các định nghĩa gốc để phục vụ đối chiếu với ResNet-LSTM & ViT-CrossAttn.
# ==============================================================================

legacy_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class VQADataset(Dataset):
    def __init__(self, path, img_dir, vocab=None, ans_vocab=None, train=True, val=True, test=False):
        with open(path, "r") as f:
            if test:
                self.data = json.load(f)["questions"]
            else:
                self.data = json.load(f)["root"]["questions"]

        self.img_dir = img_dir
        self.train = train
        self.val = val
        self.vocab = vocab
        self.ans_vocab = ans_vocab

        if vocab is None:
            self.vocab = self.build_vocab()
        if train and ans_vocab is None:
            self.ans_vocab = self.build_ans_vocab()

    def build_vocab(self):
        words = []
        for q in self.data:
            words.extend(q["question"].lower().replace("?", "").replace(";", "").split())
        counter = collections.Counter(words)
        vocab = {w: i + 2 for i, (w, _) in enumerate(counter.items())}
        vocab["<PAD>"] = 0
        vocab["<UNK>"] = 1
        return vocab

    def build_ans_vocab(self):
        answers = [q["answer"] for q in self.data if "answer" in q]
        return {a: i for i, a in enumerate(sorted(set(answers)))}

    def __len__(self):
        return len(self.data)

    def encode_q(self, q):
        q = q.lower().replace("?", "").replace(";", "").split()
        ids = [self.vocab.get(w, 1) for w in q]
        if len(ids) < Config.MAX_LEN:
            ids += [0] * (Config.MAX_LEN - len(ids))
        return torch.tensor(ids[: Config.MAX_LEN])

    def __getitem__(self, idx):
        q = self.data[idx]
        img_path = os.path.join(self.img_dir, q["image_filename"])
        img = legacy_transform(Image.open(img_path).convert("RGB"))
        q_tokens = self.encode_q(q["question"])
        if (self.train and "answer" in q) or (self.val and "answer" in q):
            return img, q_tokens, self.ans_vocab[q["answer"]]
        return img, q_tokens, q["qid"]


class Data_Loader:
    def __init__(self):
        self.vocab = Config.VOCAB
        self.ans_vocab = Config.ANS_VOCAB

    def load_data(self, path_json, img_dir, train=True, val=False, test=False):
        ds = VQADataset(
            path=path_json,
            img_dir=img_dir,
            vocab=self.vocab,
            ans_vocab=self.ans_vocab,
            train=train,
            val=val,
            test=test,
        )
        return DataLoader(
            ds,
            batch_size=Config.BATCH_SIZE,
            shuffle=train,
            num_workers=2 if IS_KAGGLE_ENV else 0,
            pin_memory=torch.cuda.is_available(),
            generator=g,
        )


class Model_ResNet_LSTM(nn.Module):
    def __init__(self, vocab_size, num_answers):
        super().__init__()
        resnet = models.resnet18(weights="IMAGENET1K_V1")
        self.cnn = nn.Sequential(*list(resnet.children())[:-2])
        self.embedding = nn.Embedding(vocab_size, 128)
        self.lstm = nn.LSTM(128, 256, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(512 + 256, 512),
            nn.ReLU(),
            nn.Linear(512, num_answers),
        )

    def forward(self, img, q):
        v = self.cnn(img).mean([2, 3])
        q = self.embedding(q)
        _, (h, _) = self.lstm(q)
        q = h.squeeze(0)
        return self.classifier(torch.cat([v, q], dim=1))


class Model_ViT_Cross_attn(nn.Module):
    def __init__(self, vocab_size, num_answers, i_hidden=196, i_embed=768, q_embed=256):
        super().__init__()
        self.vit = models.vit_b_16(weights="IMAGENET1K_V1")
        self.embed = nn.Embedding(vocab_size, q_embed, padding_idx=0)
        self.q_cls_token = nn.Parameter(torch.randn(1, 1, q_embed) * 0.02)
        self.q_pos_embed = nn.Parameter(torch.randn(1, Config.MAX_LEN + 1, q_embed) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(d_model=q_embed, nhead=8, batch_first=True)
        self.trans = nn.TransformerEncoder(encoder_layer, num_layers=4)
        self.i_layerN = nn.LayerNorm(i_embed)
        self.q_layerN = nn.LayerNorm(q_embed)
        self.linear = nn.Linear(i_embed, q_embed)
        self.cross_attn = nn.MultiheadAttention(embed_dim=q_embed, num_heads=8, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(q_embed, 512),
            nn.ReLU(),
            nn.Linear(512, num_answers),
        )
        self.cross_n1 = nn.LayerNorm(q_embed)
        self.cross_n2 = nn.LayerNorm(q_embed)
        self.ffn = nn.Sequential(
            nn.Linear(q_embed, q_embed * 4),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(q_embed * 4, q_embed),
        )

    def get_vit_token(self, img):
        with torch.no_grad():
            i = self.vit._process_input(img)
            cls_token = self.vit.class_token.expand(i.shape[0], -1, -1)
            i = torch.cat([cls_token, i], dim=1)
            i = self.vit.encoder(i)
        return self.i_layerN(i)

    def forward(self, img, q):
        i = self.linear(self.get_vit_token(img))
        pad_mask = q == 0
        q = self.embed(q)
        batch_size = q.size(0)
        cls_token = self.q_cls_token.expand(batch_size, -1, -1)
        q = torch.cat([cls_token, q], dim=1)
        cls_mask = torch.zeros(batch_size, 1, dtype=torch.bool, device=q.device)
        pad_mask = torch.cat([cls_mask, pad_mask], dim=1)
        q = q + self.q_pos_embed[:, : q.size(1), :]
        q = self.q_layerN(self.trans(q, src_key_padding_mask=pad_mask))
        q_attn, _ = self.cross_attn(query=q, key=i, value=i)
        q = self.cross_n1(q + q_attn)
        q = self.cross_n2(q + self.ffn(q))
        return self.classifier(q[:, 0, :])

## 10. Mô hình Token-Level Cross-Attention (`CLIPTokenCrossAttentionVQA`)

Nhận đặc trưng token từ Frozen CLIP:
- `Q = text_projection(text_tokens)` từ `[B, L, 512]` $\rightarrow$ `[B, L, 256]`
- `K = V = image_projection(image_patches)` từ `[B, 196, 768]` $\rightarrow$ `[B, 196, 256]`
- Qua `NUM_CROSS_LAYERS = 2` tầng `CrossAttentionBlock` (8 heads, Residual + LayerNorm + FFN `256 → 1024 → 256`), sau đó `Masked Mean Pooling` trên các token hợp lệ và `MLP Classifier` $\rightarrow$ `[B, num_answers]`.

In [ ]:
class CrossAttentionBlock(nn.Module):
    '''Một tầng Question-to-Image Cross-Attention kèm Residual, LayerNorm và FFN.'''

    def __init__(self, d_model: int, num_heads: int, ffn_dim: int, dropout: float):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.dropout1 = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(d_model)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model),
        )
        self.dropout2 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(
        self,
        q: torch.Tensor,
        kv: torch.Tensor,
        query_mask: Optional[torch.Tensor] = None,
        need_weights: bool = False,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        attn_out, attn_weights = self.cross_attn(
            query=q,
            key=kv,
            value=kv,
            need_weights=need_weights,
            average_attn_weights=False,
        )
        q = self.norm1(q + self.dropout1(attn_out))
        q = self.norm2(q + self.dropout2(self.ffn(q)))
        if query_mask is not None:
            q = q * query_mask.unsqueeze(-1).to(q.dtype)
        return q, attn_weights


class CLIPTokenCrossAttentionVQA(nn.Module):
    '''Mô hình Cross-Attention VQA nhẹ (~2M tham số) chỉ nhận token embeddings của CLIP.'''

    def __init__(
        self,
        num_answers: int,
        vision_dim: int = Config.VISION_DIM,
        text_dim: int = Config.TEXT_DIM,
        d_model: int = Config.D_MODEL,
        num_heads: int = Config.NUM_HEADS,
        num_cross_layers: int = Config.NUM_CROSS_LAYERS,
        ffn_dim: int = Config.FFN_DIM,
        dropout: float = Config.DROPOUT,
    ):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError(f"D_MODEL ({d_model}) phải chia hết cho NUM_HEADS ({num_heads}).")

        self.text_proj = nn.Sequential(
            nn.LayerNorm(text_dim),
            nn.Linear(text_dim, d_model),
            nn.Dropout(dropout),
        )
        self.image_proj = nn.Sequential(
            nn.LayerNorm(vision_dim),
            nn.Linear(vision_dim, d_model),
            nn.Dropout(dropout),
        )

        self.cross_layers = nn.ModuleList(
            [
                CrossAttentionBlock(
                    d_model=d_model,
                    num_heads=num_heads,
                    ffn_dim=ffn_dim,
                    dropout=dropout,
                )
                for _ in range(num_cross_layers)
            ]
        )

        self.final_norm = nn.LayerNorm(d_model)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, ffn_dim // 2),
            nn.LayerNorm(ffn_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim // 2, num_answers),
        )

    @staticmethod
    def masked_mean_pool(x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        mask_f = mask.unsqueeze(-1).to(x.dtype)
        denom = mask_f.sum(dim=1).clamp(min=1e-6)
        return (x * mask_f).sum(dim=1) / denom

    def forward(
        self,
        image_features: torch.Tensor,
        question_features: torch.Tensor,
        question_mask: torch.Tensor,
        return_attn: bool = False,
    ) -> Any:
        target_dtype = self.text_proj[1].weight.dtype
        q = self.text_proj(question_features.to(target_dtype))
        kv = self.image_proj(image_features.to(target_dtype))

        q = q * question_mask.unsqueeze(-1).to(q.dtype)
        attn_maps: List[torch.Tensor] = []
        for layer in self.cross_layers:
            q, attn_w = layer(q, kv, query_mask=question_mask, need_weights=return_attn)
            if return_attn and attn_w is not None:
                attn_maps.append(attn_w)

        pooled = self.masked_mean_pool(self.final_norm(q), question_mask)
        logits = self.classifier(pooled)
        if return_attn:
            return logits, attn_maps
        return logits


print("Đã định nghĩa CLIPTokenCrossAttentionVQA thành công.")

## 11. Kiểm thử Kích thước Tensor & Dòng Gradient (Shape & Gradient Smoke Tests)

Xác nhận trên 1 batch thực tế từ `train_loader` + `clip_extractor`:
1. Hình dạng `image_features` `[B, 196, 768]`, `question_features` `[B, L, 512]`, `logits` `[B, num_answers]`, `attn_maps[-1]` `[B, 8, L, 196]`.
2. Toàn bộ tham số của `CLIPTokenCrossAttentionVQA` nhận gradient hữu hạn khác 0, trong khi **0 tham số** của `CLIPVisionModel` / `CLIPTextModel` bị thay đổi hay nhận gradient.
3. Kiểm chứng bất biến padding mask (`question_mask == False`).

In [ ]:
def resolve_training_batch_size(configured_batch_size: int, device: torch.device) -> int:
    if not Config.AUTO_SCALE_BATCH_SIZE or device.type != "cuda":
        return configured_batch_size

    free_bytes, _ = torch.cuda.mem_get_info(device)
    free_gb = free_bytes / (1024 ** 3)
    if free_gb < 3.0:
        adjusted = max(16, configured_batch_size // 2)
        print(f"[VRAM GUARD] VRAM trống ({free_gb:.2f} GB) -> tạm giảm effective batch size từ {configured_batch_size} xuống {adjusted}.")
        return adjusted
    return configured_batch_size


def run_model_and_gradient_smoke_tests(
    model: CLIPTokenCrossAttentionVQA,
    extractor: BatchCLIPFeatureExtractor,
    train_loader: DataLoader,
    num_answers: int,
    device: torch.device,
) -> None:
    model.to(device)
    model.train()

    batch = next(iter(train_loader))
    img_feat, q_feat, q_mask = extractor.extract_batch(
        batch["image_keys"],
        batch["image_paths"],
        batch["questions"],
    )
    labels = batch["label"].to(device)
    B, N, Dv = img_feat.shape
    _, L, Dt = q_feat.shape

    assert N == Config.NUM_PATCHES and Dv == Config.VISION_DIM
    assert Dt == Config.TEXT_DIM and q_mask.shape == (B, L)

    logits, attn_maps = model(img_feat, q_feat, q_mask, return_attn=True)
    assert logits.shape == (B, num_answers)
    assert attn_maps[-1].shape == (B, Config.NUM_HEADS, L, Config.NUM_PATCHES)

    model.zero_grad(set_to_none=True)
    loss = nn.CrossEntropyLoss()(logits, labels)
    loss.backward()

    no_grad_params = [
        name for name, p in model.named_parameters()
        if p.requires_grad and (p.grad is None or not torch.isfinite(p.grad).all() or p.grad.abs().sum().item() == 0.0)
    ]
    assert not no_grad_params, f"Tham số không nhận gradient hợp lệ: {no_grad_params}"

    # Xác nhận 100% Frozen CLIP không có gradient
    assert all(p.grad is None and not p.requires_grad for p in extractor.vision_model.parameters())
    assert all(p.grad is None and not p.requires_grad for p in extractor.text_model.parameters())

    model.eval()
    with torch.no_grad():
        out_clean = model(img_feat, q_feat, q_mask)
        if (~q_mask).any():
            noisy_q = q_feat.clone()
            noisy_q[~q_mask] = 999.0
            out_noisy = model(img_feat, noisy_q, q_mask)
            max_diff = (out_clean - out_noisy).abs().max().item()
            assert max_diff < 1e-4, f"Padding token bị rò rỉ (max_diff={max_diff})!"
        else:
            max_diff = 0.0

    model.zero_grad(set_to_none=True)
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("[SMOKE TEST PASS] Shape, Gradient & Frozen CLIP verification:")
    print(f"  - image_features        : {tuple(img_feat.shape)}")
    print(f"  - question_features     : {tuple(q_feat.shape)}")
    print(f"  - logits                : {tuple(logits.shape)}")
    print(f"  - attn_map[-1]          : {tuple(attn_maps[-1].shape)}")
    print(f"  - Trainable parameters  : {trainable_params:,}")
    print(f"  - CLIP grads            : None (Frozen 100%)")
    print(f"  - Padding invariance Δ  : {max_diff:.2e}")


effective_batch_size = resolve_training_batch_size(Config.BATCH_SIZE, device)
train_ds, val_ds, test_ds, train_loader, val_loader, test_loader = build_on_the_fly_dataloaders(
    dataset_info=dataset_info,
    batch_size=effective_batch_size,
)

smoke_model = CLIPTokenCrossAttentionVQA(num_answers=len(dataset_info["ans_vocab"]))
run_model_and_gradient_smoke_tests(
    model=smoke_model,
    extractor=clip_extractor,
    train_loader=train_loader,
    num_answers=len(dataset_info["ans_vocab"]),
    device=device,
)
del smoke_model

## 12. Cấu hình Huấn luyện (Training Configuration)

Khởi tạo mô hình chính thức `CLIPTokenCrossAttentionVQA`, bộ tối ưu `AdamW` (chỉ tối ưu phần Cross-Attention + Projection + Classifier) và lịch learning rate `CosineAnnealingLR` kèm Linear Warmup.

In [ ]:
def build_optimizer_and_scheduler(
    model: nn.Module,
    epochs: int = Config.EPOCHS,
    lr: float = Config.LEARNING_RATE,
    weight_decay: float = Config.WEIGHT_DECAY,
) -> Tuple[optim.Optimizer, optim.lr_scheduler.LRScheduler]:
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    if not trainable_params:
        raise ValueError("Mô hình không có tham số trainable!")

    optimizer = optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    warmup_epochs = max(1, int(round(epochs * Config.WARMUP_RATIO))) if epochs >= 3 else 0
    if warmup_epochs > 0:
        warmup_sched = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.2, end_factor=1.0, total_iters=warmup_epochs)
        cosine_sched = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs - warmup_epochs), eta_min=lr * 0.01)
        scheduler = optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup_sched, cosine_sched], milestones=[warmup_epochs])
    else:
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs), eta_min=lr * 0.01)
    return optimizer, scheduler


def print_training_summary(model: nn.Module, batch_size: int) -> None:
    trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("=== CẤU HÌNH HUẤN LUYỆN THỰC NGHIỆM C (OPTION 2) ===")
    print(f"  Device                 : {device}")
    print(f"  DEBUG_MODE             : {Config.DEBUG_MODE}")
    print(f"  Train / Val / Test     : {len(train_ds):,} / {len(val_ds):,} / {len(test_ds):,}")
    print(f"  Num Answers            : {len(dataset_info['ans_vocab']):,}")
    print(f"  Batch Size (effective) : {batch_size} (Config={Config.BATCH_SIZE})")
    print(f"  Epochs / Patience      : {Config.EPOCHS} / {Config.EARLY_STOPPING_PATIENCE}")
    print(f"  LR / Weight Decay      : {Config.LEARNING_RATE} / {Config.WEIGHT_DECAY}")
    print(f"  Trainable Params       : {trainable_count:,} (CLIP frozen 100%)")
    print(f"  Checkpoint Path        : {Path.SAVE_PATH}")


main_model = CLIPTokenCrossAttentionVQA(num_answers=len(dataset_info["ans_vocab"]))
print_training_summary(main_model, effective_batch_size)

## 13. Vòng lặp Huấn luyện (`CLIPCrossAttnTrainer` / `Tranning`)

- Gọi `clip_extractor.extract_batch(...)` trong `torch.no_grad()` để trích xuất on-the-fly có khử trùng lặp + RAM LRU cache.
- Cập nhật trọng số chỉ cho `CLIPTokenCrossAttentionVQA` dưới CUDA AMP, Gradient Clipping và Learning-Rate Scheduler.
- Theo dõi thêm `ram_cache_hit_rate` và `ram_cache_mb` qua từng epoch để quan sát trực tiếp hiệu quả tiết kiệm tính toán ViT.
- Lưu checkpoint tốt nhất tại `/kaggle/working/best_clip_crossattn.pth` theo **Validation Accuracy** (dùng **Validation Loss** làm tie-breaker).

In [ ]:
class CLIPCrossAttnTrainer:
    '''Trainer On-the-Fly với khử trùng lặp ảnh trong batch và RAM LRU Cache.'''

    def __init__(
        self,
        model: CLIPTokenCrossAttentionVQA,
        extractor: BatchCLIPFeatureExtractor,
        train_loader: DataLoader,
        val_loader: DataLoader,
        epochs: int = Config.EPOCHS,
        lr: float = Config.LEARNING_RATE,
        weight_decay: float = Config.WEIGHT_DECAY,
        patience: int = Config.EARLY_STOPPING_PATIENCE,
        save_path: str = Path.SAVE_PATH,
    ):
        self.model = model
        self.extractor = extractor
        self.t_loader = train_loader
        self.v_loader = val_loader
        self.epochs = epochs
        self.patience = patience
        self.save_path = save_path

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

        self.opt, self.scheduler = build_optimizer_and_scheduler(
            model=self.model,
            epochs=self.epochs,
            lr=lr,
            weight_decay=weight_decay,
        )
        self.loss_fn = nn.CrossEntropyLoss()
        self.use_amp = self.device.type == "cuda"
        self.scaler = torch.amp.GradScaler("cuda", enabled=self.use_amp)

        self.history: Dict[str, List[float]] = {
            "epoch": [],
            "train_loss": [],
            "val_loss": [],
            "train_acc": [],
            "val_acc": [],
            "train_f1": [],
            "val_f1": [],
            "lr": [],
            "epoch_time_sec": [],
            "peak_cuda_mem_mb": [],
            "ram_cache_hit_rate": [],
            "ram_cache_mb": [],
        }

        self.best_val_acc = -1.0
        self.best_val_loss = float("inf")
        self.best_val_f1 = 0.0
        self.best_epoch = 0

    def _is_better_checkpoint(self, val_acc: float, val_loss: float) -> bool:
        if val_acc > self.best_val_acc + 1e-6:
            return True
        if abs(val_acc - self.best_val_acc) <= 1e-6 and val_loss < self.best_val_loss:
            return True
        return False

    def _evaluate_loader(self, loader: DataLoader, epoch_idx: int) -> Tuple[float, float, float]:
        self.model.eval()
        total_loss = 0.0
        correct = 0
        total_samples = 0
        all_preds: List[int] = []
        all_targets: List[int] = []

        with torch.no_grad():
            for batch in tqdm(loader, desc=f"Epoch {epoch_idx+1:02d}/{self.epochs:02d} [Val]", leave=False):
                img_feat, q_feat, q_mask = self.extractor.extract_batch(
                    batch["image_keys"], batch["image_paths"], batch["questions"]
                )
                y = batch["label"].to(self.device, non_blocking=True)

                with torch.amp.autocast("cuda", enabled=self.use_amp):
                    logits = self.model(img_feat, q_feat, q_mask)
                    loss = self.loss_fn(logits, y)

                bs = y.size(0)
                total_loss += loss.item() * bs
                preds = logits.argmax(dim=1)
                correct += (preds == y).sum().item()
                total_samples += bs
                all_preds.extend(preds.cpu().tolist())
                all_targets.extend(y.cpu().tolist())

        avg_loss = total_loss / max(total_samples, 1)
        acc = correct / max(total_samples, 1)
        f1 = float(f1_score(all_targets, all_preds, average="weighted", zero_division=0))
        return avg_loss, acc, f1

    def train(self) -> Dict[str, List[float]]:
        print(f"Bắt đầu huấn luyện trên: {self.device} (AMP={self.use_amp})")
        if self.device.type == "cuda":
            cudnn.benchmark = True

        no_improve_epochs = 0
        for epoch in range(self.epochs):
            epoch_start = time.perf_counter()
            if self.device.type == "cuda":
                torch.cuda.reset_peak_memory_stats(self.device)

            batch_sampler = getattr(self.t_loader, "batch_sampler", None)
            if hasattr(batch_sampler, "set_epoch"):
                batch_sampler.set_epoch(epoch)

            self.extractor.reset_counters()
            self.model.train()
            total_train_loss = 0.0
            train_correct = 0
            train_samples = 0
            all_train_preds: List[int] = []
            all_train_targets: List[int] = []

            pbar = tqdm(self.t_loader, desc=f"Epoch {epoch+1:02d}/{self.epochs:02d} [Train]", leave=False)
            for batch in pbar:
                img_feat, q_feat, q_mask = self.extractor.extract_batch(
                    batch["image_keys"], batch["image_paths"], batch["questions"]
                )
                y = batch["label"].to(self.device, non_blocking=True)

                self.opt.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda", enabled=self.use_amp):
                    logits = self.model(img_feat, q_feat, q_mask)
                    loss = self.loss_fn(logits, y)

                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.opt)
                nn.utils.clip_grad_norm_(self.model.parameters(), Config.GRAD_CLIP_NORM)
                self.scaler.step(self.opt)
                self.scaler.update()

                bs = y.size(0)
                total_train_loss += loss.item() * bs
                preds = logits.argmax(dim=1)
                train_correct += (preds == y).sum().item()
                train_samples += bs
                all_train_preds.extend(preds.detach().cpu().tolist())
                all_train_targets.extend(y.detach().cpu().tolist())

                pbar.set_postfix(
                    loss=f"{loss.item():.4f}",
                    acc=f"{train_correct / max(train_samples, 1):.4f}",
                    hit=f"{self.extractor.ram_lru.hit_rate*100:.0f}%",
                )

            current_lr = self.opt.param_groups[0]["lr"]
            self.scheduler.step()

            epoch_train_loss = total_train_loss / max(train_samples, 1)
            epoch_train_acc = train_correct / max(train_samples, 1)
            epoch_train_f1 = float(f1_score(all_train_targets, all_train_preds, average="weighted", zero_division=0))

            epoch_val_loss, epoch_val_acc, epoch_val_f1 = self._evaluate_loader(self.v_loader, epoch)
            epoch_time = time.perf_counter() - epoch_start
            peak_mem_mb = (
                torch.cuda.max_memory_allocated(self.device) / (1024 ** 2)
                if self.device.type == "cuda"
                else 0.0
            )
            hit_rate = self.extractor.ram_lru.hit_rate
            cache_mb = self.extractor.ram_lru.used_mb

            self.history["epoch"].append(epoch + 1)
            self.history["train_loss"].append(epoch_train_loss)
            self.history["val_loss"].append(epoch_val_loss)
            self.history["train_acc"].append(epoch_train_acc)
            self.history["val_acc"].append(epoch_val_acc)
            self.history["train_f1"].append(epoch_train_f1)
            self.history["val_f1"].append(epoch_val_f1)
            self.history["lr"].append(current_lr)
            self.history["epoch_time_sec"].append(epoch_time)
            self.history["peak_cuda_mem_mb"].append(peak_mem_mb)
            self.history["ram_cache_hit_rate"].append(hit_rate)
            self.history["ram_cache_mb"].append(cache_mb)

            if self._is_better_checkpoint(epoch_val_acc, epoch_val_loss):
                self.best_val_acc = epoch_val_acc
                self.best_val_loss = epoch_val_loss
                self.best_val_f1 = epoch_val_f1
                self.best_epoch = epoch + 1
                no_improve_epochs = 0

                torch.save(
                    {
                        "epoch": epoch + 1,
                        "model_state_dict": self.model.state_dict(),
                        "optimizer_state_dict": self.opt.state_dict(),
                        "best_val_acc": self.best_val_acc,
                        "best_val_loss": self.best_val_loss,
                        "best_val_f1": self.best_val_f1,
                        "history": self.history,
                        "model_config": {
                            "num_answers": len(dataset_info["ans_vocab"]),
                            "vision_dim": Config.VISION_DIM,
                            "text_dim": Config.TEXT_DIM,
                            "d_model": Config.D_MODEL,
                            "num_heads": Config.NUM_HEADS,
                            "num_cross_layers": Config.NUM_CROSS_LAYERS,
                            "ffn_dim": Config.FFN_DIM,
                            "dropout": Config.DROPOUT,
                            "clip_model_name": Config.MODEL_NAME,
                        },
                    },
                    self.save_path,
                )
                best_tag = " * [BEST SAVED]"
            else:
                no_improve_epochs += 1
                best_tag = ""

            print(
                f"Epoch {epoch+1:02d}/{self.epochs:02d} | "
                f"Train: L={epoch_train_loss:.4f}, Acc={epoch_train_acc:.4f}, F1={epoch_train_f1:.4f} | "
                f"Val: L={epoch_val_loss:.4f}, Acc={epoch_val_acc:.4f}, F1={epoch_val_f1:.4f} | "
                f"LRU={hit_rate*100:.0f}% ({cache_mb:.0f}MB) | {epoch_time:.1f}s | VRAM={peak_mem_mb:.1f}MB{best_tag}"
            )

            if no_improve_epochs >= self.patience:
                print(f"[EARLY STOPPING] Dừng sớm sau {self.patience} epochs không cải thiện.")
                break

        with open(Path.HISTORY_JSON_PATH, "w", encoding="utf-8") as f:
            json.dump(self.history, f, indent=2)
        print(
            f"\n[DONE] Best Val Acc: {self.best_val_acc:.4f} | Best Val Loss: {self.best_val_loss:.4f} "
            f"| Best Val F1: {self.best_val_f1:.4f} (Epoch {self.best_epoch})"
        )
        print(f"[SAVED] Checkpoint: {self.save_path}")
        return self.history


Tranning = CLIPCrossAttnTrainer


def run_overfit_sanity_test(
    extractor: BatchCLIPFeatureExtractor,
    train_loader: DataLoader,
    num_answers: int,
    device: torch.device,
    steps: int = 35,
) -> None:
    batch = next(iter(train_loader))
    n_sub = min(16, len(batch["label"]))
    img_feat, q_feat, q_mask = extractor.extract_batch(
        batch["image_keys"][:n_sub],
        batch["image_paths"][:n_sub],
        batch["questions"][:n_sub],
    )
    y = batch["label"][:n_sub].to(device)

    tmp_model = CLIPTokenCrossAttentionVQA(num_answers=num_answers, dropout=0.0).to(device)
    opt = optim.AdamW(tmp_model.parameters(), lr=1e-3, weight_decay=0.0)
    loss_fn = nn.CrossEntropyLoss()

    losses = []
    tmp_model.train()
    for _ in range(steps):
        opt.zero_grad(set_to_none=True)
        logits = tmp_model(img_feat, q_feat, q_mask)
        loss = loss_fn(logits, y)
        loss.backward()
        opt.step()
        losses.append(loss.item())

    tmp_model.eval()
    with torch.no_grad():
        final_acc = (tmp_model(img_feat, q_feat, q_mask).argmax(dim=1) == y).float().mean().item()

    print(
        f"[OVERFIT SANITY TEST] {n_sub} mẫu trong {steps} bước: "
        f"loss {losses[0]:.4f} -> {losses[-1]:.4f} | acc={final_acc:.4f}"
    )
    assert losses[-1] < losses[0], "Loss không giảm khi thử overfit tập con!"
    del tmp_model, opt


if Config.DEBUG_MODE and Config.RUN_SMOKE_OVERFIT_TEST:
    run_overfit_sanity_test(
        extractor=clip_extractor,
        train_loader=train_loader,
        num_answers=len(dataset_info["ans_vocab"]),
        device=device,
    )

trainer = CLIPCrossAttnTrainer(
    model=main_model,
    extractor=clip_extractor,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=10 if Config.DEBUG_MODE else Config.EPOCHS,
    lr=Config.LEARNING_RATE,
    weight_decay=Config.WEIGHT_DECAY,
    patience=Config.EARLY_STOPPING_PATIENCE,
    save_path=Path.SAVE_PATH,
)
history = trainer.train()

## 14. Đánh giá Mô hình & Kiểm chứng Checkpoint (Validation & Metrics)

Nạp lại trọng số tốt nhất từ `/kaggle/working/best_clip_crossattn.pth` và tính các chỉ số Loss, Top-1 Accuracy, Top-5 Accuracy, Weighted F1 và Macro F1 trên tập Validation.

In [ ]:
def evaluate_checkpoint_on_validation(
    checkpoint_path: str,
    extractor: BatchCLIPFeatureExtractor,
    val_loader: DataLoader,
    ans_vocab: Dict[str, int],
    device: torch.device,
) -> Tuple[CLIPTokenCrossAttentionVQA, Dict[str, float]]:
    assert os.path.exists(checkpoint_path), f"Không tìm thấy checkpoint tại: {checkpoint_path}"
    ckpt = load_torch_file(checkpoint_path, map_location=device)
    model_cfg = ckpt.get("model_config", {})

    eval_model = CLIPTokenCrossAttentionVQA(
        num_answers=model_cfg.get("num_answers", len(ans_vocab)),
        vision_dim=model_cfg.get("vision_dim", Config.VISION_DIM),
        text_dim=model_cfg.get("text_dim", Config.TEXT_DIM),
        d_model=model_cfg.get("d_model", Config.D_MODEL),
        num_heads=model_cfg.get("num_heads", Config.NUM_HEADS),
        num_cross_layers=model_cfg.get("num_cross_layers", Config.NUM_CROSS_LAYERS),
        ffn_dim=model_cfg.get("ffn_dim", Config.FFN_DIM),
        dropout=model_cfg.get("dropout", Config.DROPOUT),
    ).to(device)
    eval_model.load_state_dict(ckpt["model_state_dict"])
    eval_model.eval()

    loss_fn = nn.CrossEntropyLoss()
    use_amp = device.type == "cuda"
    total_loss = 0.0
    total_samples = 0
    top5_correct = 0
    all_preds: List[int] = []
    all_targets: List[int] = []

    with torch.no_grad():
        for batch in val_loader:
            img_feat, q_feat, q_mask = extractor.extract_batch(
                batch["image_keys"], batch["image_paths"], batch["questions"]
            )
            y = batch["label"].to(device, non_blocking=True)

            with torch.amp.autocast("cuda", enabled=use_amp):
                logits = eval_model(img_feat, q_feat, q_mask)
                loss = loss_fn(logits, y)

            bs = y.size(0)
            total_loss += loss.item() * bs
            total_samples += bs
            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_targets.extend(y.cpu().tolist())

            k = min(5, logits.size(1))
            topk_preds = logits.topk(k, dim=1).indices
            top5_correct += (topk_preds == y.unsqueeze(1)).any(dim=1).sum().item()

    metrics = {
        "checkpoint_epoch": int(ckpt.get("epoch", -1)),
        "val_loss": total_loss / max(total_samples, 1),
        "val_top1_acc": float(accuracy_score(all_targets, all_preds)),
        "val_top5_acc": top5_correct / max(total_samples, 1),
        "val_weighted_f1": float(f1_score(all_targets, all_preds, average="weighted", zero_division=0)),
        "val_macro_f1": float(f1_score(all_targets, all_preds, average="macro", zero_division=0)),
    }

    print("=== KẾT QUẢ ĐÁNH GIÁ CHECKPOINT TỐT NHẤT ===")
    print(f"  Checkpoint Path  : {checkpoint_path} (Epoch {metrics['checkpoint_epoch']})")
    print(f"  Val Loss         : {metrics['val_loss']:.4f}")
    print(f"  Val Top-1 Acc    : {metrics['val_top1_acc']:.4f}")
    print(f"  Val Top-5 Acc    : {metrics['val_top5_acc']:.4f}")
    print(f"  Val Weighted F1  : {metrics['val_weighted_f1']:.4f}")
    print(f"  Val Macro F1     : {metrics['val_macro_f1']:.4f}")
    return eval_model, metrics


best_model, val_metrics = evaluate_checkpoint_on_validation(
    checkpoint_path=Path.SAVE_PATH,
    extractor=clip_extractor,
    val_loader=val_loader,
    ans_vocab=dataset_info["ans_vocab"],
    device=device,
)

## 15. Trực quan hóa Lịch sử Huấn luyện (Training Visualization)

Hiển thị diễn biến Loss, Accuracy, Weighted F1-Score cùng tỷ lệ RAM Cache Hit Rate và thời gian mỗi epoch.

In [ ]:
def plot_training_history(history: Dict[str, List[float]], save_path: str = Path.HISTORY_PLOT_PATH) -> None:
    epochs = history.get("epoch", [])
    if not epochs:
        print("Chưa có dữ liệu history để vẽ.")
        return

    fig, axes = plt.subplots(1, 4, figsize=(22, 4.8))

    axes[0].plot(epochs, history["train_loss"], "o-", label="Train Loss", color="royalblue")
    axes[0].plot(epochs, history["val_loss"], "s-", label="Val Loss", color="crimson")
    axes[0].set_title("Cross-Entropy Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, linestyle="--", alpha=0.5)

    axes[1].plot(epochs, history["train_acc"], "o-", label="Train Acc", color="seagreen")
    axes[1].plot(epochs, history["val_acc"], "s-", label="Val Acc", color="darkorange")
    axes[1].set_title("Top-1 Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    axes[1].grid(True, linestyle="--", alpha=0.5)

    axes[2].plot(epochs, history["train_f1"], "o-", label="Train F1", color="purple")
    axes[2].plot(epochs, history["val_f1"], "s-", label="Val F1", color="teal")
    axes[2].set_title("Weighted F1-Score")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("F1")
    axes[2].legend()
    axes[2].grid(True, linestyle="--", alpha=0.5)

    ax_hit = axes[3]
    hit_pct = [r * 100.0 for r in history["ram_cache_hit_rate"]]
    l1 = ax_hit.plot(epochs, hit_pct, "^-", label="RAM LRU Hit (%)", color="darkgreen")
    ax_hit.set_title("RAM LRU Hit Rate & Epoch Time")
    ax_hit.set_xlabel("Epoch")
    ax_hit.set_ylabel("Hit Rate (%)", color="darkgreen")
    ax_hit.set_ylim(-5, 105)
    ax_hit.grid(True, linestyle="--", alpha=0.5)

    ax_t = ax_hit.twinx()
    l2 = ax_t.plot(epochs, history["epoch_time_sec"], "d:", label="Time (s)", color="dimgray")
    ax_t.set_ylabel("Time (sec)", color="dimgray")
    lines = l1 + l2
    ax_hit.legend(lines, [line.get_label() for line in lines], loc="center right")

    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"[SAVED] Biểu đồ huấn luyện tại: {save_path}")


plot_training_history(history, save_path=Path.HISTORY_PLOT_PATH)

## 16. Trực quan hóa Bản đồ Cross-Attention (Optional Attention Heatmap)

> ⚠️ **Lưu ý diễn giải khoa học**:
> Trọng số Cross-Attention phản ánh mức liên kết mềm (soft token-patch alignment) tại tầng chú ý được chọn và **không phải lời giải thích nhân quả trung thực (faithful causal explanation)** cho dự đoán cuối cùng.

In [ ]:
def preprocess_pil_for_clip_overlay(pil_img: Image.Image, processor: CLIPImageProcessor) -> np.ndarray:
    pixel_values = processor(images=pil_img.convert("RGB"), return_tensors="pt")["pixel_values"].squeeze(0)
    mean = torch.tensor(processor.image_mean).view(3, 1, 1)
    std = torch.tensor(processor.image_std).view(3, 1, 1)
    unnorm = (pixel_values * std + mean).clamp(0.0, 1.0)
    return unnorm.permute(1, 2, 0).numpy()


def select_visualized_token_index(tokens: List[str], valid_len: int, preferred_index: Optional[int] = None) -> Tuple[int, str]:
    if preferred_index is not None and 0 <= preferred_index < valid_len:
        return preferred_index, tokens[preferred_index]
    stopwords = {"<|startoftext|>", "<|endoftext|>", "what", "is", "the", "are", "in", "on", "of", "a", "an", "there", "how", "many", "?"}
    candidates = [
        (idx, tokens[idx].replace("</w>", "").strip().lower())
        for idx in range(valid_len)
        if tokens[idx].replace("</w>", "").strip().lower() not in stopwords and not tokens[idx].startswith("<|")
    ]
    if candidates:
        chosen_idx, _ = candidates[-1]
        return chosen_idx, tokens[chosen_idx]
    fallback_idx = min(1, max(0, valid_len - 1))
    return fallback_idx, tokens[fallback_idx]


def visualize_cross_attention_examples(
    model: CLIPTokenCrossAttentionVQA,
    extractor: BatchCLIPFeatureExtractor,
    val_ds: OnTheFlyVQADataset,
    ans_vocab: Dict[str, int],
    num_examples: int = 3,
    layer_idx: int = -1,
    token_idx_override: Optional[int] = None,
) -> None:
    os.makedirs(Path.ATTN_VIZ_DIR, exist_ok=True)
    idx2ans = {v: k for k, v in ans_vocab.items()}
    model.eval()
    grid_size = int(math.isqrt(Config.NUM_PATCHES))
    n_show = min(num_examples, len(val_ds))

    for sample_i in range(n_show):
        item = val_ds[sample_i]
        gt_ans = idx2ans[item["label"]]
        img_feat, q_feat, q_mask = extractor.extract_batch(
            [item["image_key"]], [item["image_path"]], [item["question"]]
        )

        with torch.no_grad():
            logits, attn_maps = model(img_feat, q_feat, q_mask, return_attn=True)
            pred_ans = idx2ans.get(int(logits.argmax(dim=1).item()), "<UNK>")

        attn_head_mean = attn_maps[layer_idx].squeeze(0).mean(dim=0).detach().cpu()
        enc = extractor.tokenizer(
            item["question"],
            padding="max_length",
            truncation=True,
            max_length=q_feat.size(1),
            return_tensors="pt",
        )
        raw_tokens = extractor.tokenizer.convert_ids_to_tokens(enc["input_ids"].squeeze(0).tolist())
        valid_len = int(q_mask.sum().item())
        tok_idx, tok_str = select_visualized_token_index(raw_tokens, valid_len, token_idx_override)

        patch_attn = attn_head_mean[tok_idx].reshape(1, 1, grid_size, grid_size)
        upsampled = F.interpolate(
            patch_attn.float(),
            size=(Config.IMAGE_SIZE, Config.IMAGE_SIZE),
            mode="bilinear",
            align_corners=False,
        ).squeeze().numpy()
        upsampled = (upsampled - upsampled.min()) / (upsampled.max() - upsampled.min() + 1e-8)

        with Image.open(item["image_path"]) as im:
            orig_rgb = im.convert("RGB")
            clip_rgb = preprocess_pil_for_clip_overlay(orig_rgb, extractor.image_processor)

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.8))
        ax1.imshow(orig_rgb)
        ax1.set_title("Ảnh gốc")
        ax1.axis("off")

        ax2.imshow(clip_rgb)
        ax2.imshow(upsampled, cmap="jet", alpha=0.45)
        ax2.set_title(f"Cross-Attn Overlay (token: '{tok_str}')")
        ax2.axis("off")

        fig.suptitle(
            f"Q: {item['question']}\n"
            f"GT: {gt_ans} | Pred: {pred_ans}  "
            "[Lưu ý: Attention không phải giải thích nhân quả trung thực]",
            fontsize=10.5,
        )
        plt.tight_layout()
        out_file = os.path.join(Path.ATTN_VIZ_DIR, f"val_attn_example_{sample_i+1}.png")
        plt.savefig(out_file, dpi=160, bbox_inches="tight")
        plt.show()
        plt.close(fig)
        print(f"[ATTN VIZ] Đã lưu: {out_file}")


if Config.RUN_ATTENTION_VIZ:
    visualize_cross_attention_examples(
        model=best_model,
        extractor=clip_extractor,
        val_ds=val_ds,
        ans_vocab=dataset_info["ans_vocab"],
        num_examples=Config.ATTENTION_VIZ_EXAMPLES,
    )

## 17. Dự đoán Tập Test & Tạo Tệp Submission (Test Inference & Kaggle Submission)

Chạy suy luận trên tập test với `BatchCLIPFeatureExtractor` (tự động khử trùng lặp `15,000` ảnh test qua `227,899` câu hỏi test, giữ nguyên thứ tự và giá trị `qid` ban đầu), lưu `/kaggle/working/submission.csv` (`qid,answer`) và nén vào `/kaggle/working/submission.zip`.

In [ ]:
def inference(
    model: Optional[CLIPTokenCrossAttentionVQA] = None,
    checkpoint_path: str = Path.SAVE_PATH,
    extractor: Optional[BatchCLIPFeatureExtractor] = None,
    ans_vocab: Optional[Dict[str, int]] = None,
    test_records: Optional[List[Dict[str, Any]]] = None,
    submission_path: str = Path.SUBMISSION_PATH,
    submission_zip_path: str = Path.SUBMISSION_ZIP_PATH,
) -> List[List[Any]]:
    infer_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[INFERENCE] Thiết bị: {infer_device}")

    if ans_vocab is None:
        if os.path.exists(Path.ANS_VOCAB_SAVE_PATH):
            ans_vocab = load_torch_file(Path.ANS_VOCAB_SAVE_PATH, map_location="cpu")
        elif Config.ANS_VOCAB is not None:
            ans_vocab = Config.ANS_VOCAB
        else:
            raise RuntimeError("Không tìm thấy ans_vocab để giải mã nhãn!")
    idx2ans = {v: k for k, v in ans_vocab.items()}

    ckpt = load_torch_file(checkpoint_path, map_location=infer_device) if os.path.exists(checkpoint_path) else None
    if model is None:
        model_cfg = ckpt.get("model_config", {}) if isinstance(ckpt, dict) else {}
        model = CLIPTokenCrossAttentionVQA(
            num_answers=model_cfg.get("num_answers", len(ans_vocab)),
            vision_dim=model_cfg.get("vision_dim", Config.VISION_DIM),
            text_dim=model_cfg.get("text_dim", Config.TEXT_DIM),
            d_model=model_cfg.get("d_model", Config.D_MODEL),
            num_heads=model_cfg.get("num_heads", Config.NUM_HEADS),
            num_cross_layers=model_cfg.get("num_cross_layers", Config.NUM_CROSS_LAYERS),
            ffn_dim=model_cfg.get("ffn_dim", Config.FFN_DIM),
            dropout=model_cfg.get("dropout", Config.DROPOUT),
        )
    if ckpt is not None:
        state_dict = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
        model.load_state_dict(state_dict)

    model.to(infer_device)
    model.eval()

    active_extractor = extractor if extractor is not None else clip_extractor
    active_extractor.reset_counters()
    records = test_records if test_records is not None else dataset_info["test_records"]
    infer_ds = OnTheFlyVQADataset(records, Path.TEST_IMG_DIR, "test_img", ans_vocab=None, is_test=True)
    infer_loader = DataLoader(infer_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=on_the_fly_collate_fn)

    results: List[List[Any]] = []
    with torch.no_grad():
        for batch in tqdm(infer_loader, desc="Inference (On-the-Fly Deduplicated)"):
            img_feat, q_feat, q_mask = active_extractor.extract_batch(
                batch["image_keys"], batch["image_paths"], batch["questions"]
            )
            with torch.amp.autocast("cuda", enabled=(infer_device.type == "cuda")):
                logits = model(img_feat, q_feat, q_mask)
            preds = logits.argmax(dim=1).cpu().tolist()

            for qid_val, pred_id in zip(batch["qid"], preds):
                clean_qid = qid_val.item() if torch.is_tensor(qid_val) else qid_val
                results.append([clean_qid, idx2ans[pred_id]])

    os.makedirs(os.path.dirname(submission_path), exist_ok=True)
    with open(submission_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["qid", "answer"])
        writer.writerows(results)

    with zipfile.ZipFile(submission_zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(submission_path, arcname="submission.csv")

    with open(submission_path, "r", encoding="utf-8") as f:
        rows = list(csv.reader(f))
    assert rows[0] == ["qid", "answer"]
    assert len(rows) - 1 == len(records)
    with zipfile.ZipFile(submission_zip_path, "r") as zf:
        assert "submission.csv" in zf.namelist()

    print(
        f"[INFERENCE STATS] {len(results):,} câu hỏi | "
        f"Số lần chạy CLIP Vision: {active_extractor.total_vit_forwards:,} | "
        f"RAM LRU Hit: {active_extractor.ram_lru.hit_rate*100:.1f}%"
    )
    print(f"[SUBMISSION OK] {submission_path} & {submission_zip_path}")
    print("  5 dòng đầu tiên:", results[:5])
    return results


submission_rows = inference(model=best_model)

## 18. Tổng kết Thực nghiệm & Hướng dẫn Chạy (Experiment Summary)

### 1. So sánh Kiến trúc Tổng quan

| Thành phần | Baseline 1 (`Model_ResNet_LSTM`) | Baseline 2 (`Model_ViT_Cross_attn`) | Thực nghiệm C — Kiểu 2 (`CLIPTokenCrossAttentionVQA`) |
| :--- | :--- | :--- | :--- |
| **Image Backbone** | ResNet-18 (fine-tuned) | ViT-B/16 (fine-tune 4 tầng cuối) | Frozen `openai/clip-vit-base-patch16` (`no_grad` + `FP16`) |
| **Chiến lược Ảnh** | Nạp & chạy CNN mọi câu hỏi | Nạp & chạy ViT mọi câu hỏi | **Gom batch theo ảnh + Khử trùng lặp + RAM LRU Cache** |
| **Text Encoder** | Word Embedding + LSTM | Custom Vocab + 4L Transformer | Frozen CLIP Text Transformer (`[B, L, 512]` + mask) |
| **Fusion & Pooling** | Concatenation + MLP | 1L Cross-Attn $\rightarrow$ `[CLS]` | 2L Cross-Attn + Residual + LN + FFN $\rightarrow$ Masked Mean Pooling |
| **Ổ đĩa Cache cần** | 0 GB | 0 GB | **0 GB** (không tràn giới hạn 20 GB `/kaggle/working`) |

---

### 2. Quy trình Vận hành Chuẩn trên Kaggle
1. **Smoke Test (`Config.DEBUG_MODE = True`)**: Chạy **Run All** để xác minh toàn bộ dữ liệu, shape, gradient, bất biến padding mask, overfit tập con, checkpoint và file submission trong vài chục giây.
2. **Huấn luyện Đầy đủ (`Config.DEBUG_MODE = False`)**: Đổi `Config.DEBUG_MODE = False` ở cell **Mục 3** (tùy chọn đặt `Config.RUN_ATTENTION_VIZ = False`), rồi chạy **Run All**.
3. **Tinh chỉnh RAM Cache**: Mặc định `Config.RAM_CACHE_MAX_GB = 12.0` (an toàn trên Kaggle 30 GB RAM, tự cắt không vượt quá 55% RAM khả dụng). Bạn có thể điều chỉnh tùy dung lượng RAM thực tế của máy/Kaggle.